## Imports


In [ ]:
import polars as pl
from pathlib import Path

# Data Path


In [ ]:
data_folder = Path("../data")

# Load Dataset


In [ ]:
sales_df = pl.read_parquet(
    data_folder / "sales_cleaned.parquet"
)

# Preview Dataset
sales_df.head()

## Customer Metrics


In [ ]:
customer_metrics = sales_df.group_by("customer_id").agg(
    pl.count().alias("total_orders"),
    pl.col("revenue").sum().alias("total_spent"),
    pl.col("revenue").mean().alias("avg_order_value"),
    pl.col("purchase_date").min().alias("first_purchase"),
    pl.col("purchase_date").max().alias("last_purchase")
)

customer_metrics

## Customer Lifetime


In [ ]:
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("last_purchase") - pl.col("first_purchase")
    ).dt.total_days().alias("customer_lifetime_days")
)

# Purchase Frequency


In [ ]:
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("total_orders") / (pl.col("customer_lifetime_days") + 1)
    ).alias("purchase_frequency")
)

## Customer Segments


In [ ]:
customer_metrics = customer_metrics.with_columns(
    pl.when(pl.col("total_spent") > 20000)
    .then(pl.lit("VIP"))

    .when(pl.col("total_spent") > 10000)
    .then(pl.lit("Loyal"))

    .when(pl.col("total_spent") > 5000)
    .then(pl.lit("Regular"))

    .otherwise(pl.lit("Low Value"))
    .alias("customer_type")
)

# Segment Distribution


In [ ]:
segment_distribution = customer_metrics.group_by("customer_type").agg(
    pl.count().alias("customers"),
    pl.col("total_spent").sum().alias("segment_revenue"),
).sort("segment_revenue", descending=True)

# Adding revenue_share_pct column

segment_distribution = segment_distribution.with_columns(

    (
        pl.col("segment_revenue")/pl.col("segment_revenue").sum() * 100

    ).round().alias("revenue_share_pct")

)

segment_distribution

Save the dataset


In [ ]:
customer_metrics.write_parquet(data_folder/"customer_metrics.parquet")